# Step 10 — Final Test-Set Evaluation
### Credit Risk Prediction — Lending Club Dataset

Continues from the completed **Step 9** (tuned Logistic Regression and
Decision Tree, via `GridSearchCV` with leakage-safe `imblearn` pipelines).

**Goal:** evaluate the two Step 9 tuned models — and **only** those two —
on the untouched test set.

**This section does NOT:**
- refit or tune anything using `X_test`
- use the Step 8 diagnostic `fit_resample()` arrays
- perform any additional hyperparameter tuning
- perform threshold tuning
- add Random Forest, XGBoost, SHAP, NLP, or GenAI

**Target encoding assumption** (per Step 6/7): `target == 1` is the
minority **"Bad" (defaulted) loan** class (~15%), `target == 0` is the
majority **"Good" loan** class (~85%) — this is the convention already
established when the train/test split was stratified in Step 7. If your
actual encoding differs, swap the False Positive / False Negative
interpretation in the discussion section below accordingly.

In [1]:
# Ensure imbalanced-learn is available (Colab does not ship it by default)
try:
    import imblearn
except ImportError:
    %pip install -q imbalanced-learn
    import imblearn

print("imbalanced-learn version:", imblearn.__version__)

imbalanced-learn version: 0.14.2


## 1. Mount Google Drive & retrieve Step 7 artifacts

This is a separate notebook from Steps 7–9, so it retrieves everything it
needs from Drive rather than assuming any variables are already in memory.
Reproduces the exact Step 7 split and preprocessing definition
(`random_state=42`, same feature lists), so `X_train`/`X_test` are
identical to the originals used in Step 9.

In [2]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

PROJECT_DIR = '/content/drive/MyDrive/Credit_Risk_LendingClub'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Credit_Risk_LendingClub


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)

_candidates = [
    os.path.join(PROJECT_DIR, "credit_risk_step6_final.csv"),
    "credit_risk_step6_final.csv",
    "credit_final.csv",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Step 6 CSV not found in PROJECT_DIR or the working directory. "
        "Upload/copy it there, then re-run this cell."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])
assert df.shape == (41988, 37), f"Unexpected shape: {df.shape}"

RANDOM_STATE = 42
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['target'], random_state=RANDOM_STATE
)

NON_FEATURE_COLS = ['target', 'issue_d', 'earliest_cr_line']
X_train = train_df.drop(columns=NON_FEATURE_COLS)
y_train = train_df['target'].copy()
X_test = test_df.drop(columns=NON_FEATURE_COLS)
y_test = test_df['target'].copy()

numerical_features = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
    'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'delinq_amnt',
    'pub_rec_bankruptcies', 'credit_history_months', 'loan_to_income',
    'installment_to_income', 'revol_bal_to_income', 'open_acc_ratio',
    'fico_avg', 'grade_ordinal', 'emp_length_years',
    'has_public_record', 'has_delinquency', 'pub_rec_bankruptcies_missing',
    'revol_util_missing', 'emp_length_missing', 'is_income_verified'
]
categorical_features = [
    'verification_status', 'addr_state', 'dti_bin',
    'home_ownership_grouped', 'purpose_grouped'
]
assert set(numerical_features) | set(categorical_features) == set(X_train.columns)

numeric_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipe, numerical_features),
    ('cat', categorical_pipe, categorical_features),
])
preprocessor.fit(X_train)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape, " <- untouched, used ONLY for final evaluation below")
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

X_train shape: (33590, 34)
X_test shape:  (8398, 34)  <- untouched, used ONLY for final evaluation below
y_train shape: (33590,)
y_test shape:  (8398,)


## 2. Recover the Step 9 best estimators

No model file was persisted to disk at the end of Step 9, so this section
**reproduces** the exact same `GridSearchCV` runs (identical pipelines,
identical param grids, identical `StratifiedKFold(n_splits=5,
shuffle=True, random_state=42)`, identical `scoring='average_precision'`)
on the original `X_train`/`y_train`.

This is **not** additional tuning — every random state in the chain
(train/test split, CV folds, SMOTE, PCA, model initialization) is fixed,
so this deterministically lands on the same `best_params_` /
`best_estimator_` already found in Step 9. `X_test` plays no role in this
recovery step.

*(For a production workflow, the Step 9 `best_estimator_` objects would
instead be persisted with `joblib.dump()` to Drive and loaded here
directly — worth adding as a follow-up if this notebook will be re-run
often, since it skips re-running both grid searches.)*

In [4]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
SCORING_METRIC = 'average_precision'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [5]:
# --- Logistic Regression (identical to Step 9) ---
pipeline_lr = ImbPipeline(steps=[
    ('preprocessing', clone(preprocessor)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

param_grid_lr = [
    {'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs'],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier__penalty': ['l1'], 'classifier__solver': ['liblinear'],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier__penalty': ['l2'], 'classifier__solver': ['liblinear'],
     'classifier__C': [0.01, 0.1, 1, 10]},
]

grid_lr = GridSearchCV(
    estimator=pipeline_lr, param_grid=param_grid_lr,
    scoring=SCORING_METRIC, cv=cv, n_jobs=-1, refit=True,
)
grid_lr.fit(X_train, y_train)  # X_test is never passed in here

print("Logistic Regression -- best params:", grid_lr.best_params_)
print(f"Logistic Regression -- best CV {SCORING_METRIC}: {grid_lr.best_score_:.4f}")

Logistic Regression -- best params: {'classifier__C': 10, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Logistic Regression -- best CV average_precision: 0.2931


In [ ]:
# --- Decision Tree (identical to Step 9) ---
pipeline_dt = ImbPipeline(steps=[
    ('preprocessing', clone(preprocessor)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

param_grid_dt = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [4, 8, 12, None],
    'classifier__min_samples_split': [2, 10],
    'classifier__min_samples_leaf': [1, 5],
}

grid_dt = GridSearchCV(
    estimator=pipeline_dt, param_grid=param_grid_dt,
    scoring=SCORING_METRIC, cv=cv, n_jobs=-1, refit=True,
)
grid_dt.fit(X_train, y_train)  # X_test is never passed in here

print("Decision Tree -- best params:", grid_dt.best_params_)
print(f"Decision Tree -- best CV {SCORING_METRIC}: {grid_dt.best_score_:.4f}")

## 3. Generate predictions on `X_test`

Both `best_lr` and `best_dt` are the refit `best_estimator_` pipelines
from the grid searches above (preprocessing + PCA/scaling + classifier,
**without** the SMOTE step applied at prediction time — `imblearn`
pipelines automatically skip resampling steps during `.predict()` /
`.predict_proba()`, only preprocessing/scaling/PCA/the classifier run).

`X_test` is used here for the first and only time in this notebook.

In [ ]:
best_lr = grid_lr.best_estimator_
best_dt = grid_dt.best_estimator_

y_pred_lr = best_lr.predict(X_test)
y_proba_lr = best_lr.predict_proba(X_test)[:, 1]

y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

print("Logistic Regression predictions generated:", y_pred_lr.shape, y_proba_lr.shape)
print("Decision Tree predictions generated:       ", y_pred_dt.shape, y_proba_dt.shape)

## 4. Compute test-set metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

def compute_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'PR-AUC': average_precision_score(y_true, y_proba),
    }

metrics_lr = compute_metrics(y_test, y_pred_lr, y_proba_lr)
metrics_dt = compute_metrics(y_test, y_pred_dt, y_proba_dt)

print("Logistic Regression test metrics:", metrics_lr)
print("Decision Tree test metrics:      ", metrics_dt)

## 5. Comparison table

In [9]:
comparison_df = pd.DataFrame([
    {'Model': 'Logistic Regression', **metrics_lr},
    {'Model': 'Decision Tree', **metrics_dt},
])[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']]

comparison_df = comparison_df.round(4)
comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.6550,0.2488,0.6228,0.3555,0.7082,0.3036
1,Decision Tree,0.8286,0.3203,0.1083,0.1619,0.6665,0.2432


## 6. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Logistic Regression confusion matrix (rows=actual, cols=predicted):")
print(cm_lr)
print()
print("Decision Tree confusion matrix (rows=actual, cols=predicted):")
print(cm_dt)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(cm_lr, display_labels=['Good (0)', 'Bad (1)']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Logistic Regression')
ConfusionMatrixDisplay(cm_dt, display_labels=['Good (0)', 'Bad (1)']).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Decision Tree')
plt.tight_layout()
plt.show()

## 7. Classification reports

In [11]:
print("Logistic Regression -- classification report:")
print(classification_report(y_test, y_pred_lr, target_names=['Good (0)', 'Bad (1)']))
print()
print("Decision Tree -- classification report:")
print(classification_report(y_test, y_pred_dt, target_names=['Good (0)', 'Bad (1)']))

Logistic Regression -- classification report:
              precision    recall  f1-score   support

    Good (0)       0.91      0.66      0.76      7115
     Bad (1)       0.25      0.62      0.36      1283

    accuracy                           0.66      8398
   macro avg       0.58      0.64      0.56      8398
weighted avg       0.81      0.66      0.70      8398


Decision Tree -- classification report:
              precision    recall  f1-score   support

    Good (0)       0.86      0.96      0.90      7115
     Bad (1)       0.32      0.11      0.16      1283

    accuracy                           0.83      8398
   macro avg       0.59      0.53      0.53      8398
weighted avg       0.77      0.83      0.79      8398



## 8. Why these metrics matter here

**Why accuracy alone is not sufficient**
The test set mirrors the ~85% Good / ~15% Bad class split. A model that
predicts "Good" for every single applicant would score roughly 85%
accuracy while catching **zero** defaults — accuracy rewards majority-class
correctness and hides minority-class (default) performance entirely, which
is the part of this problem that actually matters.

**What a False Negative means in this project**
A False Negative is a loan the model predicted **Good (0)** that actually
**defaulted (1)**. This is the costliest error type for a lender: the loan
is approved, the principal is disbursed, and the default results in a
direct financial loss.

**What a False Positive means in this project**
A False Positive is a loan the model predicted **Bad (1)** that would
actually have been **repaid (0)**. This is an opportunity-cost error: a
creditworthy applicant is declined (or flagged for extra scrutiny), so the
lender forgoes the interest income that loan would have earned — costly,
but not a direct capital loss the way a False Negative is.

**Why PR-AUC is particularly useful here**
PR-AUC (average precision) is computed entirely in terms of the positive
(minority, "Bad") class — precision and recall both ignore the
true-negative count. With an 85/15 split, ROC-AUC's true-negative term
makes it easy to look good without being good at catching defaults;
PR-AUC does not have that inflation, so it reflects default-detection
quality more directly.

**The precision/recall trade-off**
Raising the model's sensitivity to "Bad" (higher recall — catching more
true defaults) generally means flagging more borderline applicants as
risky, which pulls in more actually-Good applicants too (lower precision).
Conversely, being more conservative about flagging "Bad" (higher
precision) means some real defaults slip through as False Negatives
(lower recall). Given the asymmetric cost described above (False Negatives
are typically costlier than False Positives in lending), this trade-off is
a business decision, not a purely statistical one — which is also why no
threshold tuning is performed in this step.

## Test Set Evaluation — Key Findings

In [12]:
lr_row = comparison_df[comparison_df['Model'] == 'Logistic Regression'].iloc[0]
dt_row = comparison_df[comparison_df['Model'] == 'Decision Tree'].iloc[0]

print(f"Logistic Regression on X_test: Accuracy={lr_row['Accuracy']:.4f}, "
      f"Precision={lr_row['Precision']:.4f}, Recall={lr_row['Recall']:.4f}, "
      f"F1={lr_row['F1']:.4f}, ROC-AUC={lr_row['ROC-AUC']:.4f}, PR-AUC={lr_row['PR-AUC']:.4f}")
print(f"Decision Tree on X_test:       Accuracy={dt_row['Accuracy']:.4f}, "
      f"Precision={dt_row['Precision']:.4f}, Recall={dt_row['Recall']:.4f}, "
      f"F1={dt_row['F1']:.4f}, ROC-AUC={dt_row['ROC-AUC']:.4f}, PR-AUC={dt_row['PR-AUC']:.4f}")

print()
print("Metric-by-metric comparison (observed, not adjusted to any prior reported figures):")
for m in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']:
    diff = lr_row[m] - dt_row[m]
    if abs(diff) < 1e-4:
        print(f"  - {m}: effectively tied ({lr_row[m]:.4f} vs {dt_row[m]:.4f}).")
    else:
        leader = 'Logistic Regression' if diff > 0 else 'Decision Tree'
        print(f"  - {m}: {leader} higher by {abs(diff):.4f} "
              f"({lr_row[m]:.4f} vs {dt_row[m]:.4f}).")

print()
tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()
tn_dt, fp_dt, fn_dt, tp_dt = cm_dt.ravel()
print("Confusion matrix breakdown (actual Bad-loan cases in X_test = "
      f"{(y_test == 1).sum()} of {len(y_test)}):")
print(f"  - Logistic Regression: TN={tn_lr}, FP={fp_lr}, FN={fn_lr}, TP={tp_lr}")
print(f"  - Decision Tree:       TN={tn_dt}, FP={fp_dt}, FN={fn_dt}, TP={tp_dt}")

print()
print("These are the corrected, leakage-safe results produced by this pipeline. They are "
      "reported as obtained, with no adjustment toward any previously reported accuracy/F1 "
      "figures from an earlier, non-leakage-safe version of this project.")
print()
print("No single model is declared 'best' from these results alone: the metric-by-metric "
      "breakdown above shows where each model leads, and the right choice depends on which "
      "error type (missed defaults vs. declined good applicants) the deployment context "
      "weighs more heavily -- a decision this notebook deliberately does not make, since no "
      "threshold tuning or cost-weighting has been performed at this stage.")

Logistic Regression on X_test: Accuracy=0.6550, Precision=0.2488, Recall=0.6228, F1=0.3555, ROC-AUC=0.7082, PR-AUC=0.3036
Decision Tree on X_test:       Accuracy=0.8286, Precision=0.3203, Recall=0.1083, F1=0.1619, ROC-AUC=0.6665, PR-AUC=0.2432

Metric-by-metric comparison (observed, not adjusted to any prior reported figures):
  - Accuracy: Decision Tree higher by 0.1736 (0.6550 vs 0.8286).
  - Precision: Decision Tree higher by 0.0715 (0.2488 vs 0.3203).
  - Recall: Logistic Regression higher by 0.5145 (0.6228 vs 0.1083).
  - F1: Logistic Regression higher by 0.1936 (0.3555 vs 0.1619).
  - ROC-AUC: Logistic Regression higher by 0.0417 (0.7082 vs 0.6665).
  - PR-AUC: Logistic Regression higher by 0.0604 (0.3036 vs 0.2432).

Confusion matrix breakdown (actual Bad-loan cases in X_test = 1283 of 8398):
  - Logistic Regression: TN=4702, FP=2413, FN=484, TP=799
  - Decision Tree:       TN=6820, FP=295, FN=1144, TP=139

These are the corrected, leakage-safe results produced by this pipeline.